In [ ]:
#imports, necassary environments, and data upload

#!pip install kneed
#!pip install umap-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from kneed import KneeLocator
from sklearn.decomposition import PCA
import umap.umap_ as umap
from sklearn.manifold import TSNE


usvLst = pd.read_csv(
    r"File location of correct usvs.csv"
)
df = pd.read_csv(
    r"File location of all usv detections.csv"
)

In [ ]:
# get correct USV labels from rated file
valid_usvs = usvLst["0"]

# keep only matching USVs
df_filtered = df[
    df["label"].isin(valid_usvs)
]

In [ ]:
#Data Processing

# create lst of desired features
features = [
    "duration",
    "rms",
    "zcr",
    "centroid",
    "bandwidth",
    "flatness",
    "rolloff",
    "f0",
    'contrast1', 'contrast2', 'contrast3', 'contrast4','contrast5', 'contrast6', 'contrast7'
]

# slice dataframe and only keep desired features.
df_features = df_filtered[features]

# separate numeric data from non numeric for clustering
X = df_features.select_dtypes(include=["number"])
X = X.fillna(0) #if data is non, change to 0
meta = df_features.select_dtypes(exclude=["number"])

#normalize feature values
scaler = StandardScaler() #create scalar object
X_scaled = scaler.fit_transform(X) #use scalar object to normalize
#X_scaled=X

In [ ]:
#Find optimal k value
k_range = range(1, 30) # init range of possible k values
wss = [] # init list that will hold within cluster sum of squares (wss, inertia)

# for rach k value in the preset range
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10) #create kmeans obj, n_init = number of times to run kmeans with diff random starts
    km.fit(X_scaled) #run kmmeans on data
    wss.append(km.inertia_) # for the current k value, add the wss to the list


In [ ]:
#plot wss loss

plt.plot(k_range, wss, marker='o')
plt.xlabel("k")
plt.ylabel("WSS (inertia)")
plt.show()

In [ ]:
# Find the knee
kneedle = KneeLocator(k_range, wss, curve='convex', direction='decreasing') # wss shoud be decreasing with a downward curve
optimal_k = kneedle.elbow #gives elbow value

print("Optimal k:", optimal_k)

In [ ]:
#create k means object
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
# run k means on data, add cluster assignment labels to datapoints
labels = kmeans.fit_predict(X_scaled)

In [ ]:
#Dimensionality reduction (PCA) 
pca = PCA(n_components=2)   # create PCA object

X_pca = pca.fit_transform(X_scaled)   # fit PCA to data

In [ ]:
# Visualize clusters (PCA)
plt.figure(figsize=(10,5))
scatter = plt.scatter(X_pca[:,0],X_pca[:,1],c=labels,cmap='Set1',s=50)

plt.colorbar(scatter, label="Cluster")
plt.title("PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
#Dimensionality reduction (UMAP) 
# initialize UMAP object
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=8,min_dist=0.01,metric='cosine',spread=3.0)

# fit Umap model
X_umap = reducer.fit_transform(X_scaled)


In [ ]:
# Visualize clusters (UMAP)
plt.figure(figsize=(10,5))
scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=labels+1, s=50,cmap="Set1",edgecolors='black',linewidths=.5)

plt.colorbar(scatter, label="Cluster")
plt.title("UMAP")
plt.show()

In [ ]:
#Dimensionality reduction (t-SNE) 
#init t-SNE object
tsne = TSNE(n_components=2,perplexity=25,learning_rate="auto",init="pca",random_state=42)

#fit t-SNE model
X_tsne = tsne.fit_transform(X_scaled)

In [ ]:
# Visualize clusters (t-SNE)
plt.figure(figsize=(10, 5))
scatter = plt.scatter(X_tsne[:, 0],X_tsne[:, 1],c=labels+1,s=35,cmap="Set1",edgecolors='black',linewidths=.5)

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE")
plt.colorbar(scatter, label="Cluster")
#plt.legend()
plt.show()